## Парсинг данных с Python — IMDb Top 250

### Задание
С сайта IMDb Top 250 необходимо получить 250 фильмов и построить исходный набор данных в формате CSV. Для каждого фильма нужно спарсить **название, год, страну, жанр и краткое описание**.

### Источник данных
Используется страница **IMDb Top 250 Movies**. Сначала парсер получает список 250 фильмов и ссылки на их страницы, затем переходит по каждой ссылке и извлекает необходимые сведения.

IMDb Top 250 содержит 250 полнометражных фильмов; рейтинг формируется с учетом оценок пользователей и количества оценок.

### Этапы парсинга
1. Получение страницы IMDb Top 250.
2. Извлечение названий и ссылок на 250 фильмов.
3. Переход на страницы отдельных фильмов.
4. Извлечение года, страны, жанра и краткого описания.
5. Формирование `DataFrame`.
6. Сохранение результата в `imdb_top250.csv`.

### Подключение библиотек

In [ ]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import json
import time
from urllib.parse import urljoin

`requests` используется для GET-запросов, `BeautifulSoup` — для разбора HTML, `pandas` — для формирования таблицы и сохранения CSV.

### Получение страницы IMDb Top 250

In [ ]:
BASE_URL = 'https://www.imdb.com'
TOP_URL = 'https://www.imdb.com/chart/top/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/150.0 Safari/537.36',
    'Accept-Language': 'en-US,en;q=0.9'
}

page = requests.get(TOP_URL, headers=headers, timeout=30)
print('Код ответа:', page.status_code)
page.raise_for_status()
soup = bs(page.text, 'html.parser')

### Извлечение 250 фильмов
На странице Top 250 находятся ссылки вида `/title/tt.../`. Из них формируем список уникальных фильмов.

In [ ]:
movie_links = []
seen = set()

for a in soup.select('a[href*="/title/"]'):
    href = a.get('href', '')
    if '/title/' not in href:
        continue
    href = href.split('?')[0].split('#')[0]
    if not href.endswith('/'):
        href += '/'
    if href not in seen:
        seen.add(href)
        movie_links.append(urljoin(BASE_URL, href))

movie_links = movie_links[:250]
print('Количество найденных фильмов:', len(movie_links))

In [ ]:
if len(movie_links) != 250:
    raise ValueError(f'Ожидалось 250 фильмов, найдено: {len(movie_links)}')

### Функция извлечения данных со страницы фильма
IMDb размещает основные сведения о фильме в структурированных данных JSON-LD. Парсер использует эти данные, а при отсутствии отдельного поля применяет резервный поиск по HTML.

In [ ]:
def parse_movie(url, rank):
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    page_soup = bs(response.text, 'html.parser')

    title = year = description = ''
    countries = []
    genres = []

    for script in page_soup.find_all('script', type='application/ld+json'):
        try:
            data = json.loads(script.string or script.get_text())
        except (json.JSONDecodeError, TypeError):
            continue

        objects = data if isinstance(data, list) else [data]
        for obj in objects:
            if not isinstance(obj, dict) or obj.get('@type') not in ('Movie', 'TVMovie'):
                continue

            title = obj.get('name', '')
            description = obj.get('description', '')
            date_published = obj.get('datePublished', '')
            year = date_published[:4] if date_published else ''
            genres = obj.get('genre', [])
            if isinstance(genres, str):
                genres = [genres]

            raw_countries = obj.get('countryOfOrigin', [])
            if isinstance(raw_countries, dict):
                raw_countries = [raw_countries]
            for country in raw_countries:
                if isinstance(country, dict) and country.get('name'):
                    countries.append(country['name'])
                elif isinstance(country, str):
                    countries.append(country)
            break

    if not title:
        h1 = page_soup.find('h1')
        title = h1.get_text(' ', strip=True) if h1 else ''
    if not year:
        year_tag = page_soup.select_one('[class*="releaseinfo"] a')
        year = year_tag.get_text(strip=True) if year_tag else ''

    return {
        'Место': rank,
        'Название': title.strip(),
        'Год': year,
        'Страна': ', '.join(dict.fromkeys(countries)),
        'Жанр': ', '.join(dict.fromkeys(genres)),
        'Краткое описание': description.strip(),
        'Ссылка': url
    }

### Парсинг всех 250 фильмов
Между запросами добавлена небольшая пауза, чтобы снизить нагрузку на сайт.

In [ ]:
result = []

for rank, url in enumerate(movie_links, start=1):
    try:
        movie = parse_movie(url, rank)
        result.append(movie)
        print(f'{rank}/250 — {movie["Название"]}')
    except Exception as error:
        print(f'{rank}/250 — ошибка: {error}')
        result.append({
            'Место': rank, 'Название': '', 'Год': '', 'Страна': '',
            'Жанр': '', 'Краткое описание': '', 'Ссылка': url
        })
    time.sleep(0.5)

### Формирование набора данных

In [ ]:
df = pd.DataFrame(result)
df.head(10)

### Проверка количества записей и пропусков

In [ ]:
print('Количество фильмов:', len(df))
print('\nКоличество пустых значений:')
print(df[['Название', 'Год', 'Страна', 'Жанр', 'Краткое описание']].isna().sum())
print('\nКоличество пустых строк:')
print((df[['Название', 'Год', 'Страна', 'Жанр', 'Краткое описание']] == '').sum())

### Сохранение данных
Сохраняем результат в CSV с кодировкой UTF-8 с BOM, чтобы файл корректно открывался в Excel.

In [ ]:
file_name = 'imdb_top250.csv'
df.to_csv(file_name, index=False, encoding='utf-8-sig')
print(f'Файл сохранён: {file_name}')

### Итоговый результат

In [ ]:
df.info()
df